# A Temporal, Epistemic Sherlock Holmes Knowledge Graph

This notebook downloads Arthur Conan Doyle's *A Scandal in Bohemia*, segments it into scenes with a local Ollama model, extracts a constrained knowledge graph, and stores it in GestaltDB. It then asks whether Sherlock Holmes could know a proposition at a selected narrative point and returns the exact claims and rules supporting the answer.

Two clocks prevent spoilers: **valid time** represents story order, while each scene's **commit checkpoint** represents what has been revealed so far. A later revelation may describe an earlier event, but it is not visible through an earlier checkpoint. Model output is untrusted: the notebook validates identifiers, references, predicates, confidence values, paragraph coverage, and verbatim evidence quotations before writing anything.

Prerequisites: Python 3.10-3.13, `gestaltdb[leveldb]`, a running Ollama server, and the locally installed `qwen3.8:27b` model. The fast default processes at most four scenes from Chapter I. Set `FULL_STORY = True` for all three chapters.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import shutil
import sys
import tempfile
import urllib.error
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path

# Prefer this checkout over an older gestaltdb installed in the kernel.
for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if (candidate / "gestaltdb" / "epistemic.py").is_file():
        sys.path.insert(0, str(candidate.resolve()))
        break

STORY_URL = "https://sherlock-holm.es/stories/plain-text/scan.txt"
OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3.8:27b"
FULL_STORY = False
DEMO_CHAPTERS = 1
DEMO_MAX_SCENES = 4
CACHE_DIR = Path.home() / ".cache" / "gestaltdb" / "sherlock"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ENTITY_KINDS = {"Person", "Place", "Object", "Document", "Organization", "Concept"}
ID_RE = re.compile(r"^[a-z][a-z0-9-]*:[a-z0-9][a-z0-9-]*$")
PREDICATE_RE = re.compile(r"^[A-Z][A-Z0-9_]*$")
HOLMES_ID = "person:sherlock-holmes"
PROMPT_VERSION = "sherlock-kg-v1"
print(f"mode={'full story' if FULL_STORY else 'fast demo'}, model={MODEL}, cache={CACHE_DIR}")

## Local Ollama client and JSON schemas

Ollama supports a JSON Schema in the `format` field. Responses are cached by model, prompt, schema, and prompt version. Delete the cache directory or change `PROMPT_VERSION` when changing extraction semantics.

In [ ]:
def _request_json(url: str, payload: dict | None = None, timeout: int = 600) -> dict:
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Cannot reach {url}. Is Ollama running and is the story URL reachable?") from exc


def check_ollama() -> None:
    tags = _request_json(f"{OLLAMA_URL}/api/tags", timeout=15)
    names = {item.get("name") for item in tags.get("models", [])}
    if MODEL not in names:
        raise RuntimeError(f"Ollama model {MODEL!r} is unavailable. Run: ollama pull {MODEL}")


def ollama_json(system: str, prompt: str, schema: dict, purpose: str) -> dict:
    cache_input = json.dumps({
        "version": PROMPT_VERSION, "model": MODEL, "system": system,
        "prompt": prompt, "schema": schema,
    }, sort_keys=True, ensure_ascii=False).encode("utf-8")
    cache_path = CACHE_DIR / f"{purpose}-{hashlib.sha256(cache_input).hexdigest()}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        "format": schema,
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "seed": 17},
    }
    response = _request_json(f"{OLLAMA_URL}/api/chat", payload)
    try:
        result = json.loads(response["message"]["content"])
    except (KeyError, TypeError, json.JSONDecodeError) as exc:
        raise ValueError(f"Ollama returned invalid structured output for {purpose}") from exc
    cache_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    return result


SCENE_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["scenes"],
    "properties": {"scenes": {"type": "array", "items": {
        "type": "object", "additionalProperties": False,
        "required": ["title", "start_paragraph", "end_paragraph", "summary", "time_label"],
        "properties": {
            "title": {"type": "string"}, "start_paragraph": {"type": "integer", "minimum": 1},
            "end_paragraph": {"type": "integer", "minimum": 1}, "summary": {"type": "string"},
            "time_label": {"type": "string"},
        },
    }}},
}


TRIPLE_PROPERTIES = {
    "subject_id": {"type": "string"}, "predicate": {"type": "string"},
    "object_id": {"type": "string"},
}
EXTRACTION_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "required": ["entities", "events", "facts", "deductions"],
    "properties": {
        "entities": {"type": "array", "items": {"type": "object", "additionalProperties": False,
            "required": ["id", "name", "kind", "aliases"], "properties": {
                "id": {"type": "string"}, "name": {"type": "string"},
                "kind": {"type": "string", "enum": sorted(ENTITY_KINDS)},
                "aliases": {"type": "array", "items": {"type": "string"}},
            }}},
        "events": {"type": "array", "items": {"type": "object", "additionalProperties": False,
            "required": ["id", "label", "kind", "participant_ids", "location_id"], "properties": {
                "id": {"type": "string"}, "label": {"type": "string"}, "kind": {"type": "string"},
                "participant_ids": {"type": "array", "items": {"type": "string"}},
                "location_id": {"type": ["string", "null"]},
            }}},
        "facts": {"type": "array", "items": {"type": "object", "additionalProperties": False,
            "required": ["id", *TRIPLE_PROPERTIES, "polarity", "known_by", "confidence", "evidence_quote", "effective_scene_id"],
            "properties": {"id": {"type": "string"}, **TRIPLE_PROPERTIES,
                "polarity": {"type": "string", "enum": ["positive", "negative"]},
                "known_by": {"type": "array", "items": {"type": "string"}},
                "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                "evidence_quote": {"type": "string"}, "effective_scene_id": {"type": "string"},
            }}},
        "deductions": {"type": "array", "items": {"type": "object", "additionalProperties": False,
            "required": ["id", "reasoner_id", "premise_fact_ids", "conclusion", "evidence_quote"],
            "properties": {"id": {"type": "string"}, "reasoner_id": {"type": "string"},
                "premise_fact_ids": {"type": "array", "minItems": 1, "items": {"type": "string"}},
                "conclusion": {"type": "object", "additionalProperties": False,
                    "required": list(TRIPLE_PROPERTIES), "properties": TRIPLE_PROPERTIES},
                "evidence_quote": {"type": "string"},
            }}},
    },
}

## Download and deterministically identify chapters

Only scene boundaries and semantic extraction are delegated to the model. Chapter headings and paragraphs are parsed deterministically from the source text.

In [ ]:
def download_story() -> str:
    path = CACHE_DIR / "a-scandal-in-bohemia.txt"
    if not path.exists():
        request = urllib.request.Request(STORY_URL, headers={"User-Agent": "gestaltdb-notebook/1"})
        with urllib.request.urlopen(request, timeout=30) as response:
            path.write_bytes(response.read())
    return path.read_text(encoding="utf-8")


def parse_chapters(text: str) -> list[dict]:
    matches = list(re.finditer(r"(?m)^\s*CHAPTER\s+([IVX]+)\s*$", text))
    if len(matches) != 3:
        raise ValueError(f"Expected three chapter headings, found {len(matches)}")
    chapters = []
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        body = text[match.end():end].split("----------", 1)[0]
        paragraphs = [re.sub(r"\s+", " ", item).strip() for item in re.split(r"\n\s*\n", body)]
        paragraphs = [item for item in paragraphs if item]
        chapters.append({"number": index + 1, "roman": match.group(1), "paragraphs": paragraphs})
    return chapters


story_text = download_story()
chapters = parse_chapters(story_text)
selected_chapters = chapters if FULL_STORY else chapters[:DEMO_CHAPTERS]
[(chapter["number"], len(chapter["paragraphs"])) for chapter in selected_chapters]

## Segment chapters into contiguous scenes

The validator requires complete, gap-free paragraph coverage. A bad segmentation therefore fails before extraction rather than silently dropping prose.

In [ ]:
def segment_chapter(chapter: dict) -> list[dict]:
    numbered = "\n".join(f"[{i}] {text}" for i, text in enumerate(chapter["paragraphs"], 1))
    prompt = f"""Split Chapter {chapter['roman']} into coherent narrative scenes.
Use every paragraph exactly once, preserving order. Scene ranges must be contiguous,
start at paragraph 1, and end at paragraph {len(chapter['paragraphs'])}. Prefer 4-10 scenes.
time_label must quote or briefly describe only time information stated in the text; use 'unspecified' otherwise.

{numbered}"""
    raw = ollama_json("You segment narrative text without interpreting unstated facts.", prompt, SCENE_SCHEMA, f"scenes-c{chapter['number']}")
    expected_start = 1
    result = []
    for ordinal, scene in enumerate(raw.get("scenes", []), 1):
        start, end = scene["start_paragraph"], scene["end_paragraph"]
        if start != expected_start or end < start or end > len(chapter["paragraphs"]):
            raise ValueError(f"Invalid scene coverage in Chapter {chapter['roman']}: {start}-{end}")
        scene_id = f"scene:c{chapter['number']:02d}-s{ordinal:02d}"
        result.append({**scene, "id": scene_id, "chapter": chapter["number"],
                       "text": "\n\n".join(chapter["paragraphs"][start - 1:end])})
        expected_start = end + 1
    if expected_start != len(chapter["paragraphs"]) + 1:
        raise ValueError(f"Scene segmentation did not cover Chapter {chapter['roman']}")
    return result


check_ollama()
scenes = [scene for chapter in selected_chapters for scene in segment_chapter(chapter)]
if not FULL_STORY:
    scenes = scenes[:DEMO_MAX_SCENES]
for ordinal, scene in enumerate(scenes):
    scene["ordinal"] = ordinal
[(scene["id"], scene["title"], scene["start_paragraph"], scene["end_paragraph"]) for scene in scenes]

## Extract entities, events, facts, and explicit deductions

Facts must be atomic entity-to-entity propositions with verbatim evidence. `known_by` records who has acquired the information by the end of the scene. Deductions are accepted only when the prose explicitly supplies both reasoning and conclusion; their premises must refer to extracted positive facts. The local model is not allowed to use outside Sherlock Holmes knowledge.

In [ ]:
def normalized_text(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip().casefold()


def validate_id(value: str, label: str) -> None:
    if not isinstance(value, str) or not ID_RE.fullmatch(value):
        raise ValueError(f"Invalid {label} ID: {value!r}")


def validate_quote(quote: str, scene: dict, label: str) -> None:
    if not quote or normalized_text(quote) not in normalized_text(scene["text"]):
        raise ValueError(f"{label} does not contain a verbatim quotation from {scene['id']}")


def extract_scene(scene: dict, entities: dict, facts: dict, available_scene_ids: set[str]) -> dict:
    entity_catalog = [f"{key} = {value['name']} ({value['kind']})" for key, value in sorted(entities.items())]
    fact_catalog = [f"{key}: {value['subject_id']} {value['predicate']} {value['object_id']}" for key, value in sorted(facts.items())]
    prompt = f"""Extract a conservative knowledge graph from {scene['id']}: {scene['title']}.
Use only the supplied prose. Reuse catalog IDs. New IDs must be lowercase namespace:slug values.
Use person:sherlock-holmes, person:john-watson, and person:irene-adler for those recurring people.
Predicates must be uppercase relation names such as OBSERVED, LOCATED_AT, WROTE, SAID, HAS_MARKING, or KNOWS_IDENTITY.
Facts must have entity objects, not literal values. Capture negation with polarity='negative'.
known_by lists only characters who possess the fact by this scene; do not equate reader knowledge with character knowledge.
effective_scene_id is when the represented state/event begins, selected from the available scene IDs.
Every evidence_quote must be a short exact substring of this scene.
A deduction is allowed only when this prose explicitly presents the reasoner's premise-to-conclusion reasoning.
Deduction premises reference positive fact IDs from the current extraction or prior fact catalog. Do not restate its conclusion as a base fact.
Do not infer from genre conventions or outside knowledge. An empty list is better than a speculative record.

Known entities:
{chr(10).join(entity_catalog) or '(none)'}

Known facts:
{chr(10).join(fact_catalog) or '(none)'}

Available scene IDs: {', '.join(sorted(available_scene_ids))}

PROSE:
{scene['text']}"""
    raw = ollama_json("You are a literal, evidence-preserving narrative KG extractor.", prompt, EXTRACTION_SCHEMA, f"extract-{scene['id'].replace(':', '-')}")

    def missing_references(payload: dict) -> list[str]:
        declared = set(entities) | {item["id"] for item in payload["entities"]} | {item["id"] for item in payload["events"]}
        referenced = set()
        for event in payload["events"]:
            referenced.update(event["participant_ids"])
            if event["location_id"] is not None:
                referenced.add(event["location_id"])
        for fact in payload["facts"]:
            referenced.update((fact["subject_id"], fact["object_id"]))
        for deduction in payload["deductions"]:
            referenced.update((deduction["reasoner_id"], deduction["conclusion"]["subject_id"], deduction["conclusion"]["object_id"]))
        return sorted(referenced - declared)

    for repair_attempt in range(2):
        missing = missing_references(raw)
        if not missing:
            break
        repair_prompt = f"""Repair this extraction using only the same prose. These referenced IDs are undeclared: {missing}.
For each missing ID, either add an entity declaration supported by the prose, replace it with the correct catalog ID,
or remove the unsupported event/fact/deduction. Preserve exact evidence quotations and return the complete extraction.
KNOWN ENTITY IDs: {sorted(entities)}
PROSE:
{scene['text']}
EXTRACTION:
{json.dumps(raw, ensure_ascii=False)}"""
        raw = ollama_json("You repair referential integrity without adding unstated information.", repair_prompt, EXTRACTION_SCHEMA, f"repair-{repair_attempt + 1}-{scene['id'].replace(':', '-')}")
    remaining_missing = set(missing_references(raw))
    if remaining_missing:
        print(f"Dropping records with unresolved references in {scene['id']}: {sorted(remaining_missing)}")
        raw["events"] = [item for item in raw["events"] if not (set(item["participant_ids"]) | ({item["location_id"]} if item["location_id"] else set())) & remaining_missing]
        raw["facts"] = [item for item in raw["facts"] if not {item["subject_id"], item["object_id"]} & remaining_missing]
        raw["deductions"] = [item for item in raw["deductions"] if not {item["reasoner_id"], item["conclusion"]["subject_id"], item["conclusion"]["object_id"]} & remaining_missing]

    local_entities = {}
    for entity in raw["entities"]:
        validate_id(entity["id"], "entity")
        if entity["kind"] not in ENTITY_KINDS or not entity["name"].strip():
            raise ValueError(f"Invalid entity: {entity}")
        previous = entities.get(entity["id"])
        if previous and previous["kind"] != entity["kind"]:
            raise ValueError(f"Entity kind changed for {entity['id']}")
        local_entities[entity["id"]] = entity

    event_ids = set()
    known_ids = set(entities) | set(local_entities)
    for event in raw["events"]:
        validate_id(event["id"], "event")
        if not event["id"].startswith("event:") or event["id"] in event_ids:
            raise ValueError(f"Invalid or duplicate event ID: {event['id']}")
        event_ids.add(event["id"])
    all_ids = known_ids | event_ids
    for event in raw["events"]:
        references = event["participant_ids"] + ([] if event["location_id"] is None else [event["location_id"]])
        if any(item not in all_ids for item in references):
            raise ValueError(f"Event {event['id']} references an unknown entity")

    conclusion_triples = {(item["conclusion"]["subject_id"], item["conclusion"]["predicate"], item["conclusion"]["object_id"]) for item in raw["deductions"]}
    restated_conclusions = [item["id"] for item in raw["facts"] if (item["subject_id"], item["predicate"], item["object_id"]) in conclusion_triples]
    raw["facts"] = [item for item in raw["facts"] if item["id"] not in restated_conclusions]
    if restated_conclusions:
        print(f"Removing deduction conclusions duplicated as base facts in {scene['id']}: {restated_conclusions}")
    valid_fact_ids = set(facts) | {item["id"] for item in raw["facts"] if item["evidence_quote"] and normalized_text(item["evidence_quote"]) in normalized_text(scene["text"])}
    rejected_quotes = [item["id"] for item in raw["facts"] if item["id"] not in valid_fact_ids]
    raw["facts"] = [item for item in raw["facts"] if item["id"] in valid_fact_ids]
    rejected_quotes.extend(item["id"] for item in raw["deductions"] if not item["evidence_quote"] or normalized_text(item["evidence_quote"]) not in normalized_text(scene["text"]))
    raw["deductions"] = [item for item in raw["deductions"] if item["id"] not in rejected_quotes and set(item["premise_fact_ids"]) <= valid_fact_ids]
    if rejected_quotes:
        print(f"Dropping records without verbatim evidence in {scene['id']}: {rejected_quotes}")

    local_facts = {}
    for fact in raw["facts"]:
        validate_id(fact["id"], "fact")
        if not fact["id"].startswith("fact:") or fact["id"] in facts or fact["id"] in local_facts:
            raise ValueError(f"Duplicate or invalid fact ID: {fact['id']}")
        if fact["subject_id"] not in all_ids or fact["object_id"] not in all_ids:
            raise ValueError(f"Fact {fact['id']} references an unknown entity")
        if not PREDICATE_RE.fullmatch(fact["predicate"]):
            raise ValueError(f"Invalid predicate in {fact['id']}")
        if fact["effective_scene_id"] not in available_scene_ids:
            raise ValueError(f"Fact {fact['id']} has an unavailable effective scene")
        validate_quote(fact["evidence_quote"], scene, fact["id"])
        local_facts[fact["id"]] = {**fact, "revealed_scene_id": scene["id"]}

    available_facts = {**facts, **local_facts}
    deduction_ids = set()
    for deduction in raw["deductions"]:
        validate_id(deduction["id"], "deduction")
        if not deduction["id"].startswith(("deduction:", "ded:")) or deduction["id"] in deduction_ids:
            raise ValueError(f"Duplicate or invalid deduction ID: {deduction['id']}")
        deduction_ids.add(deduction["id"])
        conclusion = deduction["conclusion"]
        if conclusion["subject_id"] not in all_ids or conclusion["object_id"] not in all_ids or not PREDICATE_RE.fullmatch(conclusion["predicate"]):
            raise ValueError(f"Invalid conclusion in {deduction['id']}")
        premises = [available_facts.get(item) for item in deduction["premise_fact_ids"]]
        if any(item is None or item["polarity"] != "positive" for item in premises):
            raise ValueError(f"Deduction {deduction['id']} has missing or non-positive premises")
        if deduction["reasoner_id"] == HOLMES_ID and any(HOLMES_ID not in item["known_by"] for item in premises):
            raise ValueError(f"Holmes does not know every premise of {deduction['id']}")
        validate_quote(deduction["evidence_quote"], scene, deduction["id"])
        deduction["scene_id"] = scene["id"]

    raw["entities"] = list(local_entities.values())
    raw["facts"] = list(local_facts.values())
    return raw


entity_catalog = {}
fact_catalog = {}
extractions = {}
available_scene_ids = set()
for scene in scenes:
    available_scene_ids.add(scene["id"])
    extraction = extract_scene(scene, entity_catalog, fact_catalog, available_scene_ids)
    extractions[scene["id"]] = extraction
    for entity in extraction["entities"]:
        entity_catalog.setdefault(entity["id"], entity)
    for event in extraction["events"]:
        entity_catalog.setdefault(event["id"], {"id": event["id"], "name": event["label"], "kind": "Event", "aliases": []})
    fact_catalog.update({fact["id"]: fact for fact in extraction["facts"]})

print(f"validated {len(entity_catalog)} entities/events, {len(fact_catalog)} facts, and {sum(len(x['deductions']) for x in extractions.values())} deductions")

## Build temporal topology and epistemic claims

Each selected scene receives a synthetic monotonically increasing instant starting on 20 March 1888. Explicit textual time remains in scene properties. Actual-world claims begin at their extracted effective scene but are committed only when revealed. Facts known to Holmes are copied into one aggregate `holmes-evidence` world beginning when he acquires them. Grounded rules encode only validated, explicitly narrated Holmes deductions.

In [ ]:
try:
    from gestaltdb.epistemic import Claim
    from gestaltdb.graphdb import Edge, GraphDB, Node
    from gestaltdb.temporal import TemporalInstant
    from gestaltdb.versioning import ClaimVersionWrite, EdgeVersionWrite, NodeVersionWrite
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "This notebook requires the current GestaltDB checkout (or gestaltdb>=0.10). "
        "Select the repository kernel, restart it, and run all cells from the top."
    ) from exc

STORY_ID = "story:a-scandal-in-bohemia"
STORY_START = datetime(1888, 3, 20, tzinfo=timezone.utc)
scene_by_id = {scene["id"]: scene for scene in scenes}


def scene_time(scene_id: str) -> TemporalInstant:
    value = STORY_START + timedelta(minutes=scene_by_id[scene_id]["ordinal"])
    return TemporalInstant.from_datetime(value)


def edge_write(edge_id: str, source: str, target: str, edge_type: str, valid_from: str, **properties):
    return EdgeVersionWrite.assertion(Edge(edge_id, source, target, {"type": edge_type, **properties}), (valid_from, None))


db_path = tempfile.mkdtemp(prefix="gestaltdb_sherlock_")
graph = GraphDB.create(db_path, backend="leveldb", serializer="json")
first_time = scene_time(scenes[0]["id"])
graph.assert_world_accessibility(
    agent=HOLMES_ID, from_world="actual", to_world="holmes-evidence", kind="knowledge",
    valid_from=first_time, source=STORY_URL, provenance={"notebook": PROMPT_VERSION},
)

asserted_nodes = set()
asserted_edges = set()
checkpoints = {}
previous_scene_id = None

for scene in scenes:
    extraction = extractions[scene["id"]]
    now = scene_time(scene["id"])
    writes = []

    if STORY_ID not in asserted_nodes:
        writes.append(NodeVersionWrite.assertion(Node(STORY_ID, labels=["Story"], properties={"title": "A Scandal in Bohemia", "author": "Arthur Conan Doyle", "source": STORY_URL}), (now, None)))
        asserted_nodes.add(STORY_ID)
    chapter_id = f"chapter:{scene['chapter']:02d}"
    if chapter_id not in asserted_nodes:
        writes.append(NodeVersionWrite.assertion(Node(chapter_id, labels=["Chapter"], properties={"number": scene["chapter"]}), (now, None)))
        asserted_nodes.add(chapter_id)
        edge_id = f"edge:{STORY_ID}-has-{chapter_id}"
        writes.append(edge_write(edge_id, STORY_ID, chapter_id, "HAS_CHAPTER", now))
        asserted_edges.add(edge_id)

    writes.append(NodeVersionWrite.assertion(Node(scene["id"], labels=["Scene"], properties={
        "title": scene["title"], "summary": scene["summary"], "timeLabel": scene["time_label"],
        "ordinal": scene["ordinal"], "startParagraph": scene["start_paragraph"], "endParagraph": scene["end_paragraph"],
    }), (now, None)))
    asserted_nodes.add(scene["id"])
    writes.append(edge_write(f"edge:{chapter_id}-has-{scene['id']}", chapter_id, scene["id"], "HAS_SCENE", now))
    if previous_scene_id is not None:
        writes.append(edge_write(f"edge:{previous_scene_id}-next-{scene['id']}", previous_scene_id, scene["id"], "NEXT_SCENE", now))

    for entity in extraction["entities"]:
        if entity["id"] not in asserted_nodes:
            writes.append(NodeVersionWrite.assertion(Node(entity["id"], labels=[entity["kind"]], properties={"name": entity["name"], "aliases": entity["aliases"]}), (now, None)))
            asserted_nodes.add(entity["id"])
        mention_id = f"edge:{scene['id']}-mentions-{entity['id']}"
        if mention_id not in asserted_edges:
            writes.append(edge_write(mention_id, scene["id"], entity["id"], "MENTIONS", now))
            asserted_edges.add(mention_id)

    for event in extraction["events"]:
        if event["id"] not in asserted_nodes:
            writes.append(NodeVersionWrite.assertion(Node(event["id"], labels=["Event"], properties={"name": event["label"], "kind": event["kind"]}), (now, None)))
            asserted_nodes.add(event["id"])
        writes.append(edge_write(f"edge:{event['id']}-in-{scene['id']}", event["id"], scene["id"], "OCCURS_IN", now))
        for participant_id in event["participant_ids"]:
            writes.append(edge_write(f"edge:{participant_id}-participates-{event['id']}", participant_id, event["id"], "PARTICIPATES_IN", now))
        if event["location_id"] is not None:
            writes.append(edge_write(f"edge:{event['id']}-at-{event['location_id']}", event["id"], event["location_id"], "OCCURS_AT", now))

    for fact in extraction["facts"]:
        effective = scene_time(fact["effective_scene_id"])
        provenance = {"factId": fact["id"], "sceneId": scene["id"], "quote": fact["evidence_quote"], "model": MODEL, "promptVersion": PROMPT_VERSION}
        actual = Claim(fact["subject_id"], fact["predicate"], fact["object_id"], fact["polarity"], "source:narrative", source=scene["id"], confidence=fact["confidence"], world="actual", provenance=provenance)
        writes.append(ClaimVersionWrite.assertion(actual, (effective, None)))
        if HOLMES_ID in fact["known_by"]:
            known = Claim(fact["subject_id"], fact["predicate"], fact["object_id"], fact["polarity"], "source:holmes-evidence", source=scene["id"], confidence=fact["confidence"], world="holmes-evidence", provenance=provenance)
            writes.append(ClaimVersionWrite.assertion(known, (now, None)))
        writes.append(edge_write(f"edge:{fact['id']}", fact["subject_id"], fact["object_id"], fact["predicate"], effective, polarity=fact["polarity"], factId=fact["id"], quote=fact["evidence_quote"]))

    scene_commit = graph.commit_versions(writes, metadata={"kind": "story-scene", "sceneId": scene["id"], "model": MODEL})
    for deduction in extraction["deductions"]:
        if deduction["reasoner_id"] != HOLMES_ID:
            continue
        premises = [fact_catalog[item] for item in deduction["premise_fact_ids"]]
        graph.create_rule(deduction["id"], when=[(item["subject_id"], item["predicate"], item["object_id"]) for item in premises],
                          then=(deduction["conclusion"]["subject_id"], deduction["conclusion"]["predicate"], deduction["conclusion"]["object_id"]))
    rule_run = graph.run_rules(as_of=now, world="holmes-evidence")
    checkpoints[scene["id"]] = rule_run.commit_id or scene_commit.commit_id
    previous_scene_id = scene["id"]

print(f"database: {db_path}")
print("scene checkpoints:", checkpoints)

## Ask what Holmes could know at a scene

`answer_proposition` is deterministic and accepts an existing canonical triple. `map_question` optionally uses Ollama only to select one triple from the extracted proposition catalog; it cannot introduce entities, predicates, facts, or rules. GestaltDB evaluates `KNOWS` through the selected scene checkpoint. For a derived claim, `explain_claim` returns exact historical premise and rule versions.

In [ ]:
QUESTION_SCHEMA = {"type": "object", "additionalProperties": False,
    "required": list(TRIPLE_PROPERTIES), "properties": TRIPLE_PROPERTIES}


def proposition_catalog() -> list[dict]:
    values = [{key: fact[key] for key in TRIPLE_PROPERTIES} for fact in fact_catalog.values()]
    values.extend(deduction["conclusion"] for extraction in extractions.values() for deduction in extraction["deductions"] if deduction["reasoner_id"] == HOLMES_ID)
    return list({(item["subject_id"], item["predicate"], item["object_id"]): item for item in values}.values())


def map_question(question: str) -> dict:
    candidates = proposition_catalog()
    prompt = "Map the question to exactly one listed proposition. Do not create a new value.\nQUESTION: " + question + "\nPROPOSITIONS:\n" + json.dumps(candidates, indent=2)
    mapped = ollama_json("You select a canonical proposition without answering the question.", prompt, QUESTION_SCHEMA, "question-" + hashlib.sha256(question.encode()).hexdigest()[:16])
    if mapped not in candidates:
        raise ValueError("The question did not map exactly to an extracted proposition")
    return mapped


def _claim_ids(value) -> set[str]:
    if isinstance(value, dict):
        found = {value["claimId"]} if isinstance(value.get("claimId"), str) else set()
        return found | set().union(*(_claim_ids(item) for item in value.values()), set())
    if isinstance(value, (list, tuple)):
        return set().union(*(_claim_ids(item) for item in value), set())
    return set()


def answer_proposition(proposition: dict, at_scene_id: str) -> dict:
    if proposition not in proposition_catalog():
        raise ValueError("Use an extracted proposition; strict mode does not invent query-time facts or rules")
    checkpoint = checkpoints[at_scene_id]
    result = graph.entails(HOLMES_ID, {"subject": proposition["subject_id"], "predicate": proposition["predicate"], "object": proposition["object_id"]},
                           "KNOWS", world="actual", valid_time=scene_time(at_scene_id), through_commit=checkpoint)
    system_time = graph.get_temporal_commit(checkpoint).system_time
    proofs = []
    for claim_id in sorted(_claim_ids(result.explanation)):
        visible = graph.get_claim_as_of(claim_id, valid_time=scene_time(at_scene_id), system_time=system_time)
        if visible is None or visible.claim is None or (visible.claim.subject, visible.claim.predicate, visible.claim.object) != (proposition["subject_id"], proposition["predicate"], proposition["object_id"]):
            continue
        explanation = graph.explain_claim(claim_id, valid_time=scene_time(at_scene_id), system_time=system_time)
        if explanation is None:
            continue
        proof_nodes = []
        for node in explanation.nodes:
            if node.kind == "claim":
                claim = node.value.claim
                proof_nodes.append({"kind": "claim", "triple": [claim.subject, claim.predicate, claim.object], "agent": claim.agent, "provenance": dict(claim.provenance)})
            else:
                proof_nodes.append({"kind": "rule", "ruleId": node.value.rule_id, "when": [atom.to_tuple() for atom in node.value.when], "then": node.value.then.to_tuple()})
        proofs.append({"claimId": claim_id, "nodes": proof_nodes, "truncated": explanation.truncated})
    return {"sceneId": at_scene_id, "checkpoint": checkpoint, "status": result.status.value, "confidence": result.confidence, "proposition": proposition, "proofs": proofs}


def ask(question: str, at_scene_id: str) -> dict:
    return answer_proposition(map_question(question), at_scene_id)

In [ ]:
# Demonstrate anti-spoiler behavior with the first extracted Holmes deduction.
holmes_deductions = [deduction for scene in scenes for deduction in extractions[scene["id"]]["deductions"] if deduction["reasoner_id"] == HOLMES_ID]
if not holmes_deductions:
    print("No explicit Holmes deduction was extracted in this run. Try more scenes or inspect the cached extraction.")
else:
    deduction = holmes_deductions[0]
    conclusion = deduction["conclusion"]
    at_scene = deduction["scene_id"]
    scene_index = scene_by_id[at_scene]["ordinal"]
    if scene_index > 0:
        before_scene = scenes[scene_index - 1]["id"]
        print("BEFORE:", json.dumps(answer_proposition(conclusion, before_scene), indent=2, default=str))
    print("AFTER:", json.dumps(answer_proposition(conclusion, at_scene), indent=2, default=str))

## Inspect or visualize a historical scene

Temporal Cypher can inspect topology at the same valid and system horizons. The optional visualization is fully offline. The final cleanup cell closes the database and removes its temporary directory.

In [ ]:
selected_scene_id = scenes[-1]["id"]
selected_commit = graph.get_temporal_commit(checkpoints[selected_scene_id])
historical = graph.query(
    "MATCH (a)-[r]->(b) FOR VALID_TIME AS OF datetime($valid) FOR SYSTEM_TIME AS OF datetime($system) RETURN a, r, b LIMIT 100",
    {"valid": scene_time(selected_scene_id), "system": selected_commit.system_time},
)
print(f"{len(historical.records)} historical relationships at {selected_scene_id}")

try:
    figure = graph.visualize(
        "MATCH (a)-[r]->(b) FOR VALID_TIME AS OF datetime($valid) FOR SYSTEM_TIME AS OF datetime($system) RETURN a, r, b LIMIT 100",
        parameters={"valid": scene_time(selected_scene_id), "system": selected_commit.system_time},
    )
    figure
except Exception as exc:
    print(f"Visualization unavailable: {exc}")

In [ ]:
graph.close()
shutil.rmtree(db_path, ignore_errors=True)
print("Closed and removed the temporary GestaltDB database. Cached source/model responses remain in", CACHE_DIR)

## Scope and possible extensions

This example is intentionally evidence-grounded. `UNKNOWN` means that the selected graph and rules do not establish the proposition, not that it is false. GestaltDB's positive Horn rules do not express negated premises, defaults, probabilities, arithmetic, or arbitrary natural-language inference. Extraction quality also limits recall.

Useful separate extensions would be a reusable ontology-constrained narrative extractor, a human review UI for entity resolution and rule approval, and a probabilistic or defeasible reasoning layer that writes reviewed conclusions back as sourced GestaltDB claims. Those components should remain outside the storage library.